# PhishGuard AI 2.0 — Explainable Phishing Investigation

**IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares**  
**Prepared by: Debartha Saha**

PhishGuard AI 2.0 is a defensive educational prototype that analyses user-provided email/SMS text and URLs for potential phishing indicators. It uses transparent, rule-based detection, evidence-backed findings, and a documented heuristic risk score. It does **not** visit submitted URLs, require an internet connection, or guarantee that any message is safe.

---

## How to use
1. Install dependencies: `pip install -r requirements.txt`
2. Start Jupyter: `jupyter notebook`
3. Open this notebook and **run the cells from top to bottom** (Kernel → Restart & Run All).
4. Examine the built-in synthetic examples or call `analyze_message()` / `analyze_url()` with your own text in the manual-test cell.

## Important limitations
- This is a **rule-based educational prototype**, not a production security product.
- The score is a **heuristic indicator**, not a calibrated probability.
- A low score does **not** prove that a message is safe.
- Reputation checks and live AI are not enabled; the core analysis works offline.
- Do not submit confidential, personal, or sensitive messages to unapproved external services.


In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
# Standard-library imports used by both the engine and the notebook.

import re
import json
import ipaddress
from urllib.parse import urlparse, unquote
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import matplotlib.pyplot as plt

# Try to import the companion engine module (phishguard_engine.py).
# If it is present in the working directory the notebook will use it directly,
# keeping the logic in a single, testable location.  If the file is somehow
# absent the notebook falls back to the inline definitions in Cell 2.
try:
    from phishguard_engine import (
        analyze_message,
        analyze_url,
        extract_urls,
        print_report,
        export_report,
        SCORE_MEDIUM_THRESHOLD,
        SCORE_HIGH_THRESHOLD,
    )
    _ENGINE_SOURCE = "phishguard_engine.py"
except ImportError:
    _ENGINE_SOURCE = "inline (phishguard_engine.py not found — run Cell 2)"

print(f"PhishGuard AI 2.0 loaded.  Engine source: {_ENGINE_SOURCE}")


In [ ]:
# ── Cell 2: Inline engine definition (fallback / reference copy) ──────────────
#
# This cell redefines the complete detection engine inline so the notebook
# remains independently understandable and runnable even if phishguard_engine.py
# is not present in the working directory.
#
# When phishguard_engine.py IS present (the normal case), this cell still
# defines the same names — the import in Cell 1 simply overwrites them with
# the identical module-level objects, so there is no conflict.
#
# Detection rules are intentionally transparent and easy to extend.
# This prototype never opens, visits, or downloads submitted URLs.

# ── Thresholds ────────────────────────────────────────────────────────────────

SCORE_MEDIUM_THRESHOLD = 25
SCORE_HIGH_THRESHOLD   = 50
MAX_MESSAGE_LENGTH     = 20_000
MAX_URL_LENGTH         = 2_048
MAX_EVIDENCE_LENGTH    = 240

# ── Rule term lists ───────────────────────────────────────────────────────────

URGENT_TERMS = [
    "act now", "immediately", "urgent", "within 24 hours",
    "account will be suspended", "final warning", "verify now"
]
CREDENTIAL_TERMS = [
    "password", "passcode", "one-time password", "otp",
    "verification code", "login credentials", "sign in to verify"
]
PAYMENT_TERMS = [
    "pay now", "payment failed", "claim your refund", "processing fee",
    "bank details", "card details", "wire transfer"
]
LINK_PATTERN = re.compile(r'(?i)\b(?:https?://|www\.)[^\s<>"\']+')

# ── Internal helpers ──────────────────────────────────────────────────────────

def _finding(rule_id: str, title: str, severity: str, evidence: str,
             explanation: str, recommendation: str, points: int) -> Dict[str, Any]:
    """Build a structured finding dict with truncated evidence."""
    return {
        "rule_id":        rule_id,
        "title":          title,
        "severity":       severity,
        "evidence":       str(evidence)[:MAX_EVIDENCE_LENGTH],
        "explanation":    explanation,
        "recommendation": recommendation,
        "points":         points
    }

def _score_findings(findings: List[Dict[str, Any]]) -> int:
    """
    Sum the highest point contribution per distinct rule_id; cap at 100.
    Prevents duplicate findings from the same rule inflating the score.
    """
    best_by_rule: Dict[str, int] = {}
    for item in findings:
        key = item["rule_id"]
        best_by_rule[key] = max(best_by_rule.get(key, 0), item["points"])
    return min(100, sum(best_by_rule.values()))

def _risk_level(score: int) -> str:
    if score >= SCORE_HIGH_THRESHOLD:   return "High"
    if score >= SCORE_MEDIUM_THRESHOLD: return "Medium"
    return "Low"

def _summary(score: int, findings: List[Dict[str, Any]]) -> str:
    if not findings:
        return ("No indicators matched the current rules. This is not proof that the "
                "content is safe; the rules may miss sophisticated or novel phishing attempts.")
    titles = ", ".join(x["title"].lower() for x in findings[:3])
    return (f"{len(findings)} indicator(s) matched. "
            f"Main observations: {titles}. "
            "Review the evidence before deciding what to do.")

# ── URL extraction ────────────────────────────────────────────────────────────

def extract_urls(text: str) -> List[str]:
    """Extract URL-like strings from free text without making any network requests."""
    if not isinstance(text, str):
        return []
    urls: List[str] = []
    for match in LINK_PATTERN.findall(text):
        candidate = match.rstrip(".,);]}>\"'")
        if candidate.lower().startswith("www."):
            candidate = "http://" + candidate
        if candidate not in urls:
            urls.append(candidate)
    return urls

# ── URL analysis ──────────────────────────────────────────────────────────────

def analyze_url(url: str) -> Dict[str, Any]:
    """Analyse URL structure locally using stdlib urlparse.  Does NOT visit the URL."""
    if not isinstance(url, str):
        raise ValueError("url must be a string.")
    stripped = url.strip()
    if not stripped:
        raise ValueError("Please provide a non-empty URL.")
    if len(stripped) > MAX_URL_LENGTH:
        raise ValueError(f"URL is too long (maximum {MAX_URL_LENGTH} characters).")

    raw       = stripped
    candidate = raw if "://" in raw else "http://" + raw
    parsed    = urlparse(candidate)
    findings: List[Dict[str, Any]] = []

    # Rule URL001: unusual scheme
    if parsed.scheme not in ("http", "https"):
        findings.append(_finding(
            "URL001", "Unusual URL scheme", "medium", parsed.scheme or "(missing)",
            "The URL uses a scheme outside the expected HTTP/HTTPS web schemes.",
            "Do not open the link unless you can verify its purpose.", 20
        ))

    host = (parsed.hostname or "").lower().rstrip(".")

    if not host:
        findings.append(_finding(
            "URL002", "Hostname could not be parsed", "high", raw,
            "The URL did not contain a recognizable hostname.",
            "Treat the link as suspicious and verify it independently.", 35
        ))
    else:
        # Rule URL003: IP address host
        try:
            ipaddress.ip_address(host)
            findings.append(_finding(
                "URL003", "IP address used as hostname", "medium", host,
                "The URL uses a literal IP address instead of a conventional domain name. "
                "This can be legitimate, but warrants review.",
                "Verify the destination through a trusted source before proceeding.", 15
            ))
        except ValueError:
            pass

        # Rule URL004: punycode / IDN hostname
        if host.startswith("xn--") or ".xn--" in host:
            findings.append(_finding(
                "URL004", "Internationalized/punycode hostname", "medium", host,
                "Punycode may be used legitimately, but can also represent visually deceptive domain names.",
                "Check the exact domain spelling and the organisation\u2019s official domain.", 20
            ))

        # Rule URL005: many hostname labels
        if host.count(".") >= 4:
            findings.append(_finding(
                "URL005", "Many hostname labels", "low", host,
                "The hostname contains many dot-separated labels. This is not proof of maliciousness, "
                "but can obscure the meaningful domain.",
                "Read the hostname from right to left and verify the registrable domain.", 10
            ))

        # Rule URL006: user-info in URL
        if parsed.username is not None or parsed.password is not None:
            findings.append(_finding(
                "URL006", "User information embedded in URL", "high", parsed.netloc,
                "The URL includes user-info before the hostname, a pattern that can make "
                "the actual destination harder to notice.",
                "Do not open the link; inspect the actual hostname carefully.", 30
            ))

        # Rule URL007: security-themed hostname wording
        security_words = ("login", "verify", "secure", "account", "update")
        if host.startswith("www-") or any(w in host for w in security_words):
            findings.append(_finding(
                "URL007", "Security-themed hostname wording", "low", host,
                "The hostname contains terms often used in account or verification messages. "
                "These words are not conclusive evidence by themselves.",
                "Confirm that the domain exactly matches the service\u2019s official domain.", 8
            ))

    # Rule URL008: HTTP (not HTTPS) — only when the raw URL explicitly uses http://
    if raw.lower().startswith("http://"):
        findings.append(_finding(
            "URL008", "Connection does not use HTTPS", "low", raw,
            "The URL uses HTTP rather than HTTPS. This alone does not prove phishing.",
            "Avoid entering sensitive information on a page reached through an untrusted link.", 8
        ))

    # Rule URL009: repeated percent-encoding
    raw_path = parsed.path or ""
    if re.search(r"(?i)(%[0-9a-f]{2}){3,}", raw_path):
        findings.append(_finding(
            "URL009", "Repeated URL encoding", "low", raw_path,
            "The path contains repeated percent-encoded sequences that can make it harder to inspect.",
            "Verify the destination independently and avoid entering credentials.", 8
        ))

    score = _score_findings(findings)
    return {
        "input_type":     "url",
        "submitted_input": raw,
        "normalized_url": candidate,
        "hostname":       host or None,
        "risk_score":     score,
        "risk_level":     _risk_level(score),
        "findings":       findings,
        "summary":        _summary(score, findings),
        "timestamp_utc":  datetime.now(timezone.utc).isoformat(),
        "analysis_mode":  "local_rule_based",
        "limitations": [
            "The URL was not visited.",
            "No live reputation lookup was performed.",
            "A low score does not prove that the URL is safe."
        ]
    }

# ── Message analysis ──────────────────────────────────────────────────────────

def analyze_message(message: str, sender: Optional[str] = None,
                    subject: Optional[str] = None) -> Dict[str, Any]:
    """Analyse email/SMS text using local rules only."""
    if not isinstance(message, str):
        raise ValueError("message must be a string.")
    if not message.strip():
        raise ValueError("Please provide message text.")
    if len(message) > MAX_MESSAGE_LENGTH:
        raise ValueError(f"Message is too long (maximum {MAX_MESSAGE_LENGTH:,} characters).")

    findings: List[Dict[str, Any]] = []
    lowered  = message.lower()

    # Rule MSG001: urgency / pressure language
    urgent_hits = [t for t in URGENT_TERMS if t in lowered]
    if urgent_hits:
        findings.append(_finding(
            "MSG001", "Urgency or pressure language", "medium", ", ".join(urgent_hits),
            "The message pressures the recipient to act quickly. Urgency can appear in "
            "legitimate messages too, so consider it alongside other evidence.",
            "Pause and verify the request using an official channel you locate independently.", 12
        ))

    # Rule MSG002: credential / verification request
    credential_hits = [t for t in CREDENTIAL_TERMS if t in lowered]
    if credential_hits:
        findings.append(_finding(
            "MSG002", "Credential or verification request", "high", ", ".join(credential_hits),
            "The message mentions passwords, one-time codes, or credentials. "
            "Unsolicited requests for these details are a significant warning sign.",
            "Never share passwords or one-time codes through a message link. "
            "Visit the official service directly if needed.", 25
        ))

    # Rule MSG003: payment / financial language
    payment_hits = [t for t in PAYMENT_TERMS if t in lowered]
    if payment_hits:
        findings.append(_finding(
            "MSG003", "Payment or financial information language", "medium", ", ".join(payment_hits),
            "The message asks for payment or references sensitive financial details. "
            "This can be legitimate, but unexpected requests need verification.",
            "Confirm the request with the organisation using a known official contact method.", 15
        ))

    # Rule MSG004: embedded links
    urls = extract_urls(message)
    if urls:
        findings.append(_finding(
            "MSG004", "Link present in message", "low", ", ".join(urls[:3]),
            "The message contains one or more links. Links are common in legitimate messages "
            "too; inspect their destinations before using them.",
            "Do not click unexpected links. Navigate to the official site independently.", 5
        ))
        for url in urls[:5]:
            try:
                url_result = analyze_url(url)
                for url_finding in url_result["findings"]:
                    copied = dict(url_finding)
                    copied["rule_id"] = "EMBEDDED_" + copied["rule_id"]
                    copied["evidence"] = f"{url} \u2014 {copied['evidence']}"
                    findings.append(copied)
            except ValueError:
                continue

    # Rule MSG005: public email provider for organisational claim
    sender_text = (sender or "").strip()
    if sender_text and "@" in sender_text:
        sender_domain  = sender_text.rsplit("@", 1)[-1].lower()
        consumer_domains = {"gmail.com", "yahoo.com", "outlook.com", "hotmail.com"}
        brand_keywords   = ("bank", "university", "college", "delivery",
                             "microsoft", "google", "paypal")
        if sender_domain in consumer_domains and any(b in lowered for b in brand_keywords):
            findings.append(_finding(
                "MSG005", "Public email provider used for an organisation claim", "low", sender_text,
                "The sender uses a public email provider while the message appears to represent "
                "an organisation. This can happen legitimately, but should be checked.",
                "Confirm the sender using the organisation\u2019s official website or contact details.", 10
            ))

    score = _score_findings(findings)
    return {
        "input_type":    "email_or_sms",
        "subject":       (subject or "")[:200],
        "sender":        sender_text[:320] if sender_text else "",
        "risk_score":    score,
        "risk_level":    _risk_level(score),
        "findings":      findings,
        "summary":       _summary(score, findings),
        "extracted_urls": urls,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "analysis_mode": "local_rule_based",
        "limitations": [
            "No external reputation lookup was performed.",
            "Message wording alone cannot establish sender identity.",
            "A low score does not prove that the message is safe."
        ]
    }

# ── Report helpers ────────────────────────────────────────────────────────────

def print_report(result: Dict[str, Any]) -> None:
    """Print a readable investigation report to stdout."""
    print("=" * 68)
    print("PHISHGUARD AI 2.0 \u2014 INVESTIGATION REPORT")
    print("=" * 68)
    print(f"Type    : {result['input_type']}")
    print(f"Risk    : {result['risk_level']} ({result['risk_score']}/100)")
    print(f"Time UTC: {result['timestamp_utc']}")
    print(f"Summary : {result['summary']}")
    print("\nFindings:")
    if not result["findings"]:
        print("  (No indicators matched the current rules.)")
    for item in result["findings"]:
        print(f"\n  [{item['severity'].upper()}] {item['title']}  "
              f"(rule {item['rule_id']}, +{item['points']} pts)")
        print(f"  Evidence          : {item['evidence']}")
        print(f"  Why it matters    : {item['explanation']}")
        print(f"  Recommended action: {item['recommendation']}")
    print("\nLimitations:")
    for limitation in result["limitations"]:
        print(f"  - {limitation}")
    print("=" * 68)

def export_report(result: Dict[str, Any], path: str) -> None:
    """Write a result dict to a UTF-8 JSON file."""
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(result, fh, indent=2, ensure_ascii=False)

print("\u2713 Detection engine ready.")


## Detection rules reference

| Rule ID | Scope | Title | Severity | Points |
|---------|-------|-------|----------|--------|
| MSG001 | Message | Urgency or pressure language | Medium | 12 |
| MSG002 | Message | Credential or verification request | High | 25 |
| MSG003 | Message | Payment or financial information language | Medium | 15 |
| MSG004 | Message | Link present in message | Low | 5 |
| MSG005 | Message | Public email provider for an organisation claim | Low | 10 |
| URL001 | URL | Unusual URL scheme | Medium | 20 |
| URL002 | URL | Hostname could not be parsed | High | 35 |
| URL003 | URL | IP address used as hostname | Medium | 15 |
| URL004 | URL | Internationalized/punycode hostname | Medium | 20 |
| URL005 | URL | Many hostname labels | Low | 10 |
| URL006 | URL | User information embedded in URL | High | 30 |
| URL007 | URL | Security-themed hostname wording | Low | 8 |
| URL008 | URL | Connection does not use HTTPS | Low | 8 |
| URL009 | URL | Repeated URL encoding | Low | 8 |

URL rules found within embedded links appear as `EMBEDDED_URL001`, etc., in message results.  
The score is the sum of the highest contribution per distinct rule ID, capped at 100.

**Risk categories:**
- **Low:** 0–24
- **Medium:** 25–49
- **High:** 50–100


In [ ]:
# ── Cell 3: Synthetic demonstration examples ───────────────────────────────────
# All examples are synthetic.  They are not real user messages.

examples = [
    {
        "name":    "Suspicious delivery message",
        "sender":  "delivery-update@example-mail.test",
        "message": (
            "URGENT: Your package is on hold. Verify now within 24 hours at "
            "http://delivery.example.test.verify-account.invalid/login "
            "to avoid suspension."
        )
    },
    {
        "name":    "Credential request",
        "sender":  "security-alert@gmail.com",
        "message": (
            "Your account will be suspended. Sign in to verify and enter "
            "your password and one-time password immediately."
        )
    },
    {
        "name":    "Legitimate college announcement (synthetic)",
        "sender":  "office@college.example",
        "message": (
            "Hello students, the library will open at 9 AM on Monday. "
            "Please check the notice board for details."
        )
    },
    {
        "name":    "Ambiguous reminder",
        "sender":  "events@example.org",
        "message": "Reminder: please review the event information when convenient."
    }
]

demo_rows    = []
demo_results = []

for item in examples:
    result = analyze_message(item["message"], sender=item["sender"], subject=item["name"])
    demo_results.append((item["name"], result))
    demo_rows.append({
        "example":       item["name"],
        "risk_score":    result["risk_score"],
        "risk_level":    result["risk_level"],
        "finding_count": len(result["findings"]),
        "summary":       result["summary"]
    })

demo_df = pd.DataFrame(demo_rows)

# display() is available in Jupyter; fall back to print() elsewhere.
try:
    from IPython.display import display
    display(demo_df)
except ImportError:
    print(demo_df.to_string(index=False))


In [ ]:
# ── Cell 4: Risk-score visualization ──────────────────────────────────────────
# Horizontal bar chart — easier to read with long example names.

fig, ax = plt.subplots(figsize=(9, 3.5))

colors = [
    "#d9534f" if score >= SCORE_HIGH_THRESHOLD
    else ("#f0ad4e" if score >= SCORE_MEDIUM_THRESHOLD else "#5cb85c")
    for score in demo_df["risk_score"]
]

bars = ax.barh(demo_df["example"], demo_df["risk_score"], color=colors, edgecolor="white")

# Add score labels inside/outside bars.
for bar, score in zip(bars, demo_df["risk_score"]):
    ax.text(
        score + 1, bar.get_y() + bar.get_height() / 2,
        str(score), va="center", fontsize=10
    )

ax.axvline(SCORE_MEDIUM_THRESHOLD, color="orange", linestyle="--", linewidth=1,
           label=f"Medium threshold ({SCORE_MEDIUM_THRESHOLD})")
ax.axvline(SCORE_HIGH_THRESHOLD,   color="red",    linestyle="--", linewidth=1,
           label=f"High threshold ({SCORE_HIGH_THRESHOLD})")

ax.set_xlim(0, 105)
ax.set_xlabel("Heuristic risk score (0\u2013100)")
ax.set_title("PhishGuard AI 2.0 \u2014 Heuristic risk scores for synthetic examples")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("These scores are heuristic indicators, not calibrated probabilities.")
print("Green = Low  |  Orange = Medium  |  Red = High")


In [ ]:
# ── Cell 5: Detailed report for each synthetic example ────────────────────────

for name, result in demo_results:
    print(f"\n{'='*68}")
    print(f"EXAMPLE: {name}")
    print_report(result)


In [ ]:
# ── Cell 6: Manual test — edit these values and re-run this cell ───────────────

sample_message = (
    "Your account will be suspended. Verify now and enter your password "
    "at http://example.invalid/account"
)
sample_sender = "security@example.invalid"

result = analyze_message(sample_message, sender=sample_sender)
print_report(result)

# ── URL-only analysis ─────────────────────────────────────────────────────────
# Uncomment the lines below to analyse a URL directly:
#
# url_result = analyze_url("http://user:pw@192.0.2.1/login")
# print_report(url_result)


In [ ]:
# ── Cell 7: Export a result as JSON (local file) ───────────────────────────────
# The first synthetic example is exported.  Change demo_results[0] to export
# a different result.

result_to_export = demo_results[0][1]
output_path      = "phishguard_investigation.json"

export_report(result_to_export, output_path)
print(f"Saved report to: {output_path}")

# Verify the exported file is valid JSON and print the top-level keys.
with open(output_path, encoding="utf-8") as fh:
    loaded = fh.read()
parsed_check = json.loads(loaded)
print(f"Export verified. Top-level keys: {list(parsed_check.keys())}")
print(f"Risk score in export: {parsed_check['risk_score']}  |  Risk level: {parsed_check['risk_level']}")


## Running the automated test suite

The project includes a test suite at `tests/test_phishguard.py` covering 96 test cases.

Run from the project root:

```bash
python -m pytest tests/ -v
```

Or using the built-in unittest runner:

```bash
python -m unittest discover -s tests -v
```

All 96 tests pass on Python 3.14 and are expected to pass on Python 3.10 or newer.


## Suggested future improvements

- Add an optional, explicitly enabled LLM provider to generate contextual explanations grounded in existing findings; keep the deterministic findings authoritative.
- Add opt-in threat-intelligence providers only after reviewing their current API documentation, quotas, and data-sharing policies.
- Add a persistent case database and investigation dashboard if the project scope permits.
- Expand the rule set and evaluate it against a labeled, appropriately licensed dataset.
- Add support for PDF export.

**Safety note:** Never use this tool as the sole basis for a security decision. Do not submit sensitive content to external services without explicit permission.